In [10]:
# Cell 1 - Imports
import requests
import pandas as pd
import time

In [11]:
# Cell 2 — Configuration (only this cell needs to be updated)
BASE_URL = "https://findata-qa.worldbank.org:9047"
TOKEN    = "3/Cw8MfRR5CgJxP9cfVlot3cd95gGBfZvwEa7jp3tBOU9UWfFCJ8e6k6KAhF0w=="
SCHEMA   = ["publishFINDATAREP", "IFCTRE", "Application", "Murex"]

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# ── All your queries ──
QUERIES = {
    "perf_attribution"      : 'SELECT * FROM "Perf_Attribution"',
    "perf_benchmark"        : 'SELECT * FROM "Perf_Benchmark_Weights"',
    "perf_vs_funding"       : 'SELECT * FROM "Perf_Vs_Funding_Snap"',
    "plvar_snapshot"        : 'SELECT * FROM "Plvar_Snapshot"',
    "portfolio_details"     : 'SELECT * FROM "Portfolio_Details"',
    "securities_detail"     : 'SELECT * FROM "Securities_Detail"',
    "trade_mktval_detail"   : 'SELECT * FROM "Trade_Mktval_Detail"',
    "security_prices_detail": 'SELECT * FROM "Security_Prices_Detail"',
    "inv_pnl_summary"       : 'SELECT * FROM "Inv_Pnl_Summary"',
    "inv_security_position" : 'SELECT * FROM "Inv_Security_Position"',  # ← new
}

# ── Only change this line to switch query ──
ACTIVE_QUERY = "Inv_Pnl_Summary"
SQL_QUERY    = QUERIES[ACTIVE_QUERY]


KeyError: 'Inv_Pnl_Summary'

In [6]:
# Cell 3 — Display function
def display_df(df):
    return df.style.set_properties(**{
        'white-space': 'nowrap',
        'font-size': '11px',
        'text-align': 'left',
        'border': '1px solid lightgrey',
        'color': 'black'
    }).set_table_styles([
        {
            'selector': 'thead th',
            'props': [
                ('white-space', 'nowrap'),
                ('font-size', '11px'),
                ('font-weight', 'bold'),
                ('background-color', '#4472C4'),
                ('color', 'white'),
                ('text-align', 'center'),
                ('border', '1px solid lightgrey')
            ]
        },
        {
            'selector': 'tbody tr:nth-child(even)',
            'props': [('background-color', '#f2f2f2'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:nth-child(odd)',
            'props': [('background-color', 'white'), ('color', 'black')]
        },
        {
            'selector': 'tbody tr:hover',
            'props': [('background-color', '#d6e4f0'), ('color', 'black')]
        }
    ])


In [7]:
# Cell 4 — Submit the SQL job
response = requests.post(
    f"{BASE_URL}/api/v3/sql",
    headers=HEADERS,
    json={
        "sql": SQL_QUERY,
        "context": SCHEMA
    },
    verify=True,
    timeout=30
)
response.raise_for_status()
job_id = response.json().get("id")
print(f"Job submitted successfully.")
print(f"Job ID: {job_id}")


Job submitted successfully.
Job ID: 158229dc-afe9-82af-4392-137b52784000


In [8]:
# Cell 5 — Poll until job completes
url = f"{BASE_URL}/api/v3/job/{job_id}"
for attempt in range(30):
    response = requests.get(url, headers=HEADERS, verify=True)
    response.raise_for_status()
    status = response.json().get("jobState")
    print(f"Attempt {attempt + 1}: {status}")
    if status == "COMPLETED":
        print("Job completed!")
        break
    elif status in ("FAILED", "CANCELED"):
        print(f"Job failed: {response.json().get('errorMessage')}")
        break
    time.sleep(2)


Attempt 1: COMPLETED
Job completed!


In [9]:
# Cell 6 — Fetch results and build DataFrame

# ── Column selection per query (add/remove columns as needed) ──
COLUMNS = {
    "perf_attribution"      : [], 
    "perf_benchmark"        : ['Benchmark_Name','Security_ID','Closing_date','MX_DATA_DATE','CUSIP','ISIN','Weight','Clean_Price','MX_ACCRUED','Market_value','NB_Shares'],  
    "perf_vs_funding"       : [],   # ← fill in
    "plvar_snapshot"        : [],   # ← fill in
    "portfolio_details"     : [],   # ← fill in
    "securities_detail"     : [],   # ← fill in
    "trade_mktval_detail"   : [],   # ← fill in
    "security_prices_detail": [],   # ← fill in
    "inv_pnl_summary"       : [],   # ← fill in
    "inv_security_position" : [],   # ← fill in
}

# ── Fetch first page ──
response = requests.get(
    f"{BASE_URL}/api/v3/job/{job_id}/results?offset=0&limit=500",
    headers=HEADERS,
    verify=True
)
response.raise_for_status()
data = response.json()

columns  = [col["name"] for col in data.get("schema", [])]
all_rows = data.get("rows", [])
total    = data.get("rowCount", None)

# ── Paginate if more than 500 rows ──
offset = 500
while True:
    r = requests.get(
        f"{BASE_URL}/api/v3/job/{job_id}/results?offset={offset}&limit=500",
        headers=HEADERS,
        verify=True
    )
    batch = r.json().get("rows", [])
    if not batch:
        break
    all_rows.extend(batch)
    offset += len(batch)
    print(f"Fetched {offset} rows so far...")

df = pd.DataFrame(all_rows, columns=columns)
# Compact version
print(df.columns.tolist())


# ── Apply column filter based on active query ──
selected_cols = COLUMNS.get(ACTIVE_QUERY, [])
if selected_cols:
    missing = [c for c in selected_cols if c not in df.columns]
    if missing:
        print(f"⚠️  Columns not found and skipped: {missing}")
    df = df[[c for c in selected_cols if c in df.columns]]

print(f"Active Query  : {ACTIVE_QUERY}")
print(f"Total Rows    : {len(df)}")
print(f"Total Columns : {len(df.columns)}")
print(f"Columns       : {list(df.columns)}")

Fetched 681 rows so far...
['Business_Date', 'MX_REF_DATA', 'MX_REF_JOB', 'Clean_Price', 'Closing_date', 'CUSIP', 'MX_DATA_DATE', 'Data_Label', 'Data_Generation_Date', 'Data_Generation_Desk', 'Data_Generation_Group', 'Data_Generation_Time', 'Data_Generation_User', 'ISIN', 'Benchmark_Name', 'Market_value', 'NB_Shares', 'MX_NOTATION', 'Security_ID', 'Weight', 'MX_ACCRUED', 'UUID', 'FeedName', 'ProcessedTimeStamp', 'AccountingDate', 'BatchID', 'LastProcessedBatchId']
Active Query  : perf_benchmark
Total Rows    : 681
Total Columns : 11
Columns       : ['Benchmark_Name', 'Security_ID', 'Closing_date', 'MX_DATA_DATE', 'CUSIP', 'ISIN', 'Weight', 'Clean_Price', 'MX_ACCRUED', 'Market_value', 'NB_Shares']


In [7]:
df.columns

Index(['Benchmark_Name', 'Security_ID', 'Closing_date', 'MX_DATA_DATE',
       'CUSIP', 'ISIN', 'Weight', 'Clean_Price', 'MX_ACCRUED', 'Market_value',
       'NB_Shares'],
      dtype='object')

In [8]:
display_df(df.head(10))   # first 10 rows

,Benchmark_Name,Security_ID,Closing_date,MX_DATA_DATE,CUSIP,ISIN,Weight,Clean_Price,MX_ACCRUED,Market_value,NB_Shares
0,Dummy,Dummy,2026-07-17,2026-07-17,Dummy,Dummy,0.000000,0.000000,0.000000,0.000000,0.000000
1,Dummy,Dummy,2026-07-21,2026-07-21,Dummy,Dummy,0.000000,0.000000,0.000000,0.000000,0.000000
2,ICE U.S. Trsry 1-3 Yr Bond Index,T 0.375 300927,2026-08-03,2026-08-03,91282CAL5,US91282CAL54,0.009199,95.740000,0.129098,54279.730000,56617.290000
3,ICE U.S. Trsry 1-3 Yr Bond Index,T 6.375 150827,2026-08-03,2026-08-03,912810FA1,US912810FA17,0.001639,102.170000,2.993785,9671.790000,9196.760000
4,ICE U.S. Trsry 1-3 Yr Bond Index,T 6.125 151127,2026-08-03,2026-08-03,912810FB9,US912810FB99,0.003872,102.390000,1.348166,22844.670000,22021.340000
5,ICE U.S. Trsry 1-3 Yr Bond Index,T 5.5 150828,2026-08-03,2026-08-03,912810FE3,US912810FE39,0.002095,102.390000,2.582873,12362.350000,11776.200000
6,ICE U.S. Trsry 1-3 Yr Bond Index,T 5.25 151128,2026-08-03,2026-08-03,912810FF0,US912810FF04,0.001916,102.120000,1.155571,11305.320000,10947.050000
7,ICE U.S. Trsry 1-3 Yr Bond Index,T 5.25 150229,2026-08-03,2026-08-03,912810FG8,US912810FG86,0.002020,102.540000,2.465470,11918.370000,11350.340000
8,ICE U.S. Trsry 1-3 Yr Bond Index,T 4.875 311028,2026-08-03,2026-08-03,91282CJF9,US91282CJF95,0.009005,101.230000,1.271739,53136.460000,51839.330000
9,ICE U.S. Trsry 1-3 Yr Bond Index,T 4.625 300928,2026-08-03,2026-08-03,91282CJA0,US91282CJA09,0.008495,100.700000,1.592213,50124.690000,48999.970000


In [9]:
display_df(df.tail(10))   # last 10 rows

,Benchmark_Name,Security_ID,Closing_date,MX_DATA_DATE,CUSIP,ISIN,Weight,Clean_Price,MX_ACCRUED,Market_value,NB_Shares
283,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.375 301127,2026-08-05,2026-08-05,91282CPL9,US91282CPL99,0.012944,99.010000,0.617828,76492.100000,76779.530000
284,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.75 310529,2026-08-05,2026-08-05,91282CES6,US91282CES61,0.007551,96.080000,0.503415,44620.660000,46198.120000
285,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.375 150529,2026-08-05,2026-08-05,9128286T2,US9128286T26,0.013633,95.180000,0.535666,80566.540000,84173.060000
286,ICE U.S. Trsry 1-3 Yr Bond Index,T 1.375 311028,2026-08-05,2026-08-05,91282CDF5,US91282CDF59,0.010766,94.000000,0.366168,63624.060000,67425.320000
287,ICE U.S. Trsry 1-3 Yr Bond Index,T 4.625 300429,2026-08-05,2026-08-05,91282CKP5,US91282CKP58,0.012149,100.990000,1.231658,71794.160000,70232.310000
288,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.625 150828,2026-08-05,2026-08-05,91282CNU1,US91282CNU17,0.013182,98.900000,1.722376,77898.960000,77418.330000
289,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.375 150927,2026-08-05,2026-08-05,91282CLL3,US91282CLL36,0.009960,99.200000,1.320652,58855.900000,58553.790000
290,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.25 300629,2026-08-05,2026-08-05,91282CEV9,US91282CEV90,0.007101,97.320000,0.326766,41963.000000,42974.150000
291,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.375 310329,2026-08-05,2026-08-05,91282CEE7,US91282CEE75,0.009064,95.380000,0.830601,53562.200000,55674.720000
292,ICE U.S. Trsry 1-3 Yr Bond Index,T 1.125 310828,2026-08-05,2026-08-05,91282CCV1,US91282CCV19,0.011055,93.960000,0.486073,65331.650000,69172.800000


In [10]:
# Cell 7 — Dispatch DataFrame by Closing_date

dispatched = {
    date: group.reset_index(drop=True)
    for date, group in df.groupby("MX_DATA_DATE")
}

# Preview available dates
print(f"Unique Closing Dates : {len(dispatched)}")
for date, frame in dispatched.items():
    print(f"  {date} → {len(frame)} rows")


Unique Closing Dates : 5
  2026-07-17 → 1 rows
  2026-07-21 → 1 rows
  2026-08-03 → 97 rows
  2026-08-04 → 97 rows
  2026-08-05 → 97 rows


In [11]:
df_20260805 = dispatched["2026-08-05"]
df_20260804 = dispatched["2026-08-04"]
df_20260803 = dispatched["2026-08-03"]

In [12]:
display_df(df_20260805.tail(10)) 

,Benchmark_Name,Security_ID,Closing_date,MX_DATA_DATE,CUSIP,ISIN,Weight,Clean_Price,MX_ACCRUED,Market_value,NB_Shares
87,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.375 301127,2026-08-05,2026-08-05,91282CPL9,US91282CPL99,0.012944,99.010000,0.617828,76492.100000,76779.530000
88,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.75 310529,2026-08-05,2026-08-05,91282CES6,US91282CES61,0.007551,96.080000,0.503415,44620.660000,46198.120000
89,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.375 150529,2026-08-05,2026-08-05,9128286T2,US9128286T26,0.013633,95.180000,0.535666,80566.540000,84173.060000
90,ICE U.S. Trsry 1-3 Yr Bond Index,T 1.375 311028,2026-08-05,2026-08-05,91282CDF5,US91282CDF59,0.010766,94.000000,0.366168,63624.060000,67425.320000
91,ICE U.S. Trsry 1-3 Yr Bond Index,T 4.625 300429,2026-08-05,2026-08-05,91282CKP5,US91282CKP58,0.012149,100.990000,1.231658,71794.160000,70232.310000
92,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.625 150828,2026-08-05,2026-08-05,91282CNU1,US91282CNU17,0.013182,98.900000,1.722376,77898.960000,77418.330000
93,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.375 150927,2026-08-05,2026-08-05,91282CLL3,US91282CLL36,0.009960,99.200000,1.320652,58855.900000,58553.790000
94,ICE U.S. Trsry 1-3 Yr Bond Index,T 3.25 300629,2026-08-05,2026-08-05,91282CEV9,US91282CEV90,0.007101,97.320000,0.326766,41963.000000,42974.150000
95,ICE U.S. Trsry 1-3 Yr Bond Index,T 2.375 310329,2026-08-05,2026-08-05,91282CEE7,US91282CEE75,0.009064,95.380000,0.830601,53562.200000,55674.720000
96,ICE U.S. Trsry 1-3 Yr Bond Index,T 1.125 310828,2026-08-05,2026-08-05,91282CCV1,US91282CCV19,0.011055,93.960000,0.486073,65331.650000,69172.800000


In [ ]:
# Cell 8 — Optional: Save to CSV or Excel
df.to_csv("output.csv", index=False)
print("Saved to output.csv")

df.to_excel("output.xlsx", index=False)
print("Saved to output.xlsx")
